<a href="https://colab.research.google.com/github/amenidhawadi/deep-learning/blob/main/04_Transformer_tests_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

In [ ]:
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
n_samples = 5000
n_features = 20
test_size = 0.2
val_size = 0.2

In [ ]:
X = np.random.randn(n_samples, n_features)

In [ ]:
true_weights = np.random.randn(n_features)
y_prob = 1 / (1 + np.exp(-np.dot(X, true_weights) + 0.5 * np.random.randn(n_samples)))
y = (y_prob > 0.5).astype(int)

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=test_size, random_state=42, stratify=y
)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=val_size, random_state=42, stratify=y_train_full
)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [ ]:
print(f"Formes : Train {X_train.shape}, Val {X_val.shape}, Test {X_test.shape}")



Formes : Train (3200, 20), Val (800, 20), Test (1000, 20)


In [ ]:
def transformer_tabular(d_model=64, num_heads=4, ff_dim=128, dropout=0.1, lr=1e-3):
    inputs = layers.Input(shape=(X_train.shape[1],))
    # Projection de chaque feature vers d_model
    x = layers.Dense(d_model)(inputs)
    x = layers.LayerNormalization()(x)

In [ ]:
def transformer_tabular(d_model=64, num_heads=4, ff_dim=128, dropout=0.1, lr=1e-3):
    inputs = layers.Input(shape=(X_train.shape[1],))
    # Projection
    x = layers.Dense(d_model)(inputs)
    x = layers.LayerNormalization()(x)

    # Self-attention
    attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model // num_heads)
    attn_out = attn(x, x)           # <-- cette ligne doit être alignée avec la précédente
    x = layers.Add()([x, attn_out])
    x = layers.LayerNormalization()(x)

    # Feed-forward
    ffn = keras.Sequential([
        layers.Dense(ff_dim, activation='relu'),
        layers.Dense(d_model)
    ])
    ffn_out = ffn(x)
    x = layers.Add()([x, ffn_out])
    x = layers.LayerNormalization()(x)

    # Pooling et sortie
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy', 'AUC']
    )
    return model

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
n_samples, n_features = 5000, 20
X = np.random.randn(n_samples, n_features)
true_weights = np.random.randn(n_features)
y_prob = 1 / (1 + np.exp(-np.dot(X, true_weights) + 0.5 * np.random.randn(n_samples)))
y = (y_prob > 0.5).astype(int)

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full)

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"Forme de X_train : {X_train.shape}")

Forme de X_train : (3200, 20)


In [ ]:
def transformer_tabular(d_model=64, num_heads=4, ff_dim=128, dropout=0.1, lr=1e-3):
    inputs = keras.Input(shape=(X_train.shape[1],))

In [ ]:
import os
print("Current working directory:", os.getcwd())
print("Does ../data/ exist?", os.path.exists("../data"))
print("Contents of parent directory:", os.listdir("..") if os.path.exists("..") else "N/A")

Current working directory: /content
Does ../data/ exist? False
Contents of parent directory: ['media', 'proc', 'libx32', 'usr', 'sbin', 'run', 'opt', 'lib', 'dev', 'mnt', 'root', 'lib32', 'var', 'boot', 'srv', 'sys', 'lib64', 'etc', 'bin', 'home', 'tmp', 'kaggle', '.dockerenv', 'tools', 'datalab', 'content', 'python-apt', 'python-apt.tar.xz']


In [ ]:
# If data is in a sibling folder 'data' next to your notebook folder:
X_train = np.load("../data/X_train.npy")

# If data is in the same folder as the notebook:
X_train = np.load("./data/X_train.npy")   # or just "data/X_train.npy"

# If data is in a parent's parent folder:
X_train = np.load("../../data/X_train.npy")

In [ ]:
class TransformerTabular(nn.Module):
    def __init__(self, num_features, d_model=32, nhead=4, num_layers=3, dim_feedforward=64, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(1, d_model)  # projeter chaque feature individuellement
        self.pos_encoding = nn.Parameter(torch.randn(1, num_features, d_model))
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(d_model, 1)

    def forward(self, x):
        # x: (batch, num_features, 1)
        x = self.input_proj(x)  # (batch, num_features, d_model)
        x = x + self.pos_encoding
        x = self.transformer(x)  # (batch, num_features, d_model)
        # Pooling global (moyenne sur les features)
        x = x.mean(dim=1)        # (batch, d_model)
        return self.fc_out(x)